# How to solve stochastic programs with SPAROW

**This notebook contains a demonstration for solving a simple stochastic programming exemplar with SPAROW**

In [1]:
### Import the farmer problem exemplar from sparow_examples repository
'''
    If solving your own model, see https://github.com/sandialabs/sparow_examples/ for examples of structuring the application data, 
    scenario data, Pyomo model builder(s), and stochastic programming model object (including specifying a bundling scheme).
'''
from sparow_examples.farmers.MRPfarmers import Basic_farmers, Advanced_farmers
from sparow.ef.ef import ExtensiveFormSolver
import pprint

Alternative solutions package from or_topas is available.


## Solving optimization problems in sparow_examples

In [2]:
# SP model objects are imported from sparow_examples
sp_basic = Basic_farmers()
sp_advanced = Advanced_farmers()

In [3]:
solver = ExtensiveFormSolver() 

# Solve each model object and print results, compare optimal value and solutions
solver.set_options(
    solver="gurobi_direct",   # solving with gurobi
    # max_iterations=2,  # this will default to 100 (only for PH)
    loglevel="INFO",   # can replace with DEBUG, VERBOSE, etc.
    # rho_updates=True,  # rho parameter will update at each iteration (only for PH)
)

# Return solution objects 
results_basic = solver.solve(sp_basic, solver = "gurobi_direct")
results_advanced = solver.solve(sp_advanced, solver = "gurobi_direct")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


In [4]:
# Convert results to dictionaries for easier readability and comparison
results_basic = results_basic.to_dict()
results_advanced = results_advanced.to_dict()

results_dict = {"Basic": results_basic, "Advanced": results_advanced}

for model_type, results in results_dict.items():
    print(f"\n\n ===== {model_type} ===== \n\n")
    
    for outer_key in results.keys():

        print(f"\n\n Outer Key: {outer_key}")
        innerdict = results[outer_key]

        for inner_key in innerdict.keys():
            print(f"  Inner Key: {inner_key}")
            print(f"{innerdict[inner_key]}")
            print("\n")



 ===== Basic ===== 




 Outer Key: metadata
  Inner Key: context_name
None


  Inner Key: policy
keep_best


  Inner Key: as_solution_source
sparow.solnpool.solnpool._sparow_as_solution


  Inner Key: termination_condition
optimal


  Inner Key: status
ok


  Inner Key: start_time
2026-07-07 10:46:30.377400


  Inner Key: end_time
2026-07-07 10:46:30.414156


  Inner Key: time_elapsed
0:00:00.036756




 Outer Key: solutions
  Inner Key: 0
{'id': 0, 'variables': [{'value': 170.0, 'fixed': False, 'name': 'DevotedAcreage[WHEAT]', 'index': 0, 'discrete': False, 'suffix': {}}, {'value': 80.0, 'fixed': False, 'name': 'DevotedAcreage[CORN]', 'index': 1, 'discrete': False, 'suffix': {}}, {'value': 250.0, 'fixed': False, 'name': 'DevotedAcreage[SUGAR_BEETS]', 'index': 2, 'discrete': False, 'suffix': {}}], 'objectives': [{'value': -108390.0, 'name': None, 'index': None, 'suffix': {}}], 'suffix': {}}




 Outer Key: pool_config
  Inner Key: max_pool_size
None


  Inner Key: objective_index
0


## Confidence Intervals for Upper Bound on Optimality Gap

In [5]:
from sparow_examples.farmers.MRPfarmers import (
    get_basic_ci_problem_adapter,
    get_advanced_ci_problem_adapter,
)

from sparow.ci.mrp_options import MRPOptions
from sparow.ci.standard_mrp import StandardMRP
from sparow.ci.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator

In [6]:
basic_adapter = get_basic_ci_problem_adapter(use_integer=False)
advanced_adapter = get_advanced_ci_problem_adapter(use_integer=False)

basic_scenarios = basic_adapter.get_scenario_population()
advanced_scenarios = advanced_adapter.get_scenario_population()

print("Number of Basic scenarios:", len(basic_scenarios))
print("Number of Advanced scenarios:", len(advanced_scenarios))
print("First Advanced scenario:", advanced_scenarios[0])

Number of Basic scenarios: 3
Number of Advanced scenarios: 1000
First Advanced scenario: {'ID': 'scen_0', 'Yield': {'WHEAT': 2.0, 'CORN': 2.4, 'SUGAR_BEETS': 16.0}, 'Probability': 0.001}


In [7]:
basic_obj = basic_adapter.get_objective_value(results_basic)
advanced_obj = advanced_adapter.get_objective_value(results_advanced)

basic_xhat = basic_adapter.get_first_stage_solution(results_basic)
advanced_xhat = advanced_adapter.get_first_stage_solution(results_advanced)

print("Basic true optimal value:", basic_obj)
print("Basic xhat (first-stage vars):", basic_xhat)

print("Advanced true optimal value:", advanced_obj)
print("Advanced xhat (first-stage vars):", advanced_xhat)

Basic true optimal value: -108390.0
Basic xhat (first-stage vars): {'DevotedAcreage[WHEAT]': 170.0, 'DevotedAcreage[CORN]': 80.0, 'DevotedAcreage[SUGAR_BEETS]': 250.0}
Advanced true optimal value: -110505.53571428556
Advanced xhat (first-stage vars): {'DevotedAcreage[WHEAT]': 133.03571428571428, 'DevotedAcreage[CORN]': 85.71428571428572, 'DevotedAcreage[SUGAR_BEETS]': 281.25}


In [8]:
# For quick sanity check - use the first-stage solution found with the basic farmers problem
# as the candidate solution for the advanced farmers problem
true_gap_eval = TrueOptimalityGapEvaluator(
    problem_adapter=advanced_adapter,
    scenarios=advanced_scenarios,
    solver_name="gurobi_direct",
)

advanced_true_gap_results = true_gap_eval.compute_true_gap(basic_xhat)

advanced_true_gap_results

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


{'true_optimal_value': -110505.53571428556,
 'xhat_true_value': -108549.99999999953,
 'true_gap': 1955.5357142860303}

In [9]:
# Set the options for the MRP algorithm
mrp_options = MRPOptions(
    n=100,
    m=20,
    alpha=0.05,
    seed=12345,
    with_replacement=True,
    solver_name="gurobi_direct",
    verbose=True,
)

# Instantiate the MRP object with the advanced farmers problem
mrp = StandardMRP(
    problem_adapter=advanced_adapter,
    scenarios=advanced_scenarios,
    options=mrp_options,
)

# For quick sanity check - use the first-stage solution found with the basic farmers problem
# as the candidate solution for the advanced farmers problem
mrp_results = mrp.run(xhat=basic_xhat)

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Running MRP replication 1/20


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for replication 1: F_nk = 1900.4821428571304
Running MRP replication 2/20


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


...

Gap estimate for replication 18: F_nk = 2410.599999999904
Running MRP replication 19/20


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for replication 19: F_nk = 2006.9714285714726
Running MRP replication 20/20


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


Gap estimate for replication 20: F_nk = 1639.8678571428463


In [10]:
for key, value in mrp_results.items():
    print(f"{key}: {value}")        

point_estimate: 2064.881563852798
sample_variance: 113556.44005015436
sample_std: 336.98136454432364
t_statistic: 1.7291328115213682
half_width: 130.29244642113932
ci_lower: 0.0
ci_upper: 2195.1740102739373
replication_values: [1900.48214286 2124.18571429 1957.76428571 1745.475      1720.68214286
 2018.61071429 2199.4        1546.6969697  1821.25714286 2740.36785714
 2051.98214286 2720.22142857 2524.10357143 1908.65454545 1837.74642857
 2329.13333333 2093.42857143 2410.6        2006.97142857 1639.86785714]
sampled_indices_by_replication: [[699, 227, 788, 316, 204, 797, 642, 676, 988, 391, 839, 332, 567, 598, 213, 186, 229, 672, 613, 941, 706, 248, 914, 948, 732, 667, 130, 95, 266, 441, 72, 886, 474, 697, 212, 326, 115, 733, 774, 220, 714, 81, 391, 159, 744, 340, 473, 465, 475, 266, 558, 815, 498, 193, 24, 129, 81, 91, 122, 598, 807, 854, 653, 601, 331, 931, 640, 724, 733, 860, 702, 929, 541, 546, 251, 937, 552, 494, 316, 273, 633, 451, 568, 665, 899, 330, 681, 903, 458, 257, 290, 339, 

In [11]:
print("Exact true gap:", advanced_true_gap_results["true_gap"])
print("MRP point estimate:", mrp_results["point_estimate"])
print("MRP CI upper bound:", mrp_results["ci_upper"])

Exact true gap: 1955.5357142860303
MRP point estimate: 2064.881563852798
MRP CI upper bound: 2195.1740102739373
